<a href="https://colab.research.google.com/github/AISHWARYAARULSELVAN/IPCV/blob/main/EXP_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import files
uploaded = files.upload()

In [3]:
#!/usr/bin/env python
import cv2
import glob
import numpy as np
import os
from google.colab.patches import cv2_imshow  # for Colab display

# Inputs
num_squares_x = 8  # Number of chessboard squares along x-axis
num_squares_y = 10 # Number of chessboard squares along y-axis
num_interior_corners_x = num_squares_x - 1
num_interior_corners_y = num_squares_y - 1
checker_width = 0.020  # Checker width in meters

# Path where your images are uploaded in Colab
path = "/content"  # All uploaded files go here in Colab

def calibrate():
    # Termination criteria for corner refinement
    termination_criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER,
                            30, 0.001)

    # 3D points in real world space
    object_points_3D = np.zeros((num_interior_corners_x*num_interior_corners_y, 3), np.float32)
    object_points_3D[:, :2] = np.mgrid[0:num_interior_corners_y, 0:num_interior_corners_x].T.reshape(-1, 2)
    object_points_3D *= checker_width

    object_points = []  # 3D points
    image_points = []   # 2D points

    # Read all PNG images starting with CAM
    images = glob.glob(os.path.join(path, 'CAM*.PNG'))
    print("Found images:", images)

    image_size = None

    for image_file in images:
        image = cv2.imread(image_file)
        if image is None:
            print("Skipping unreadable image:", image_file)
            continue

        gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

        # Find chessboard corners
        success, corners = cv2.findChessboardCorners(gray,
                                                     (num_interior_corners_y, num_interior_corners_x),
                                                     None)
        if success:
            object_points.append(object_points_3D)

            # Refine corner locations
            corners_2 = cv2.cornerSubPix(gray, corners, (11, 11), (-1, -1), termination_criteria)
            image_points.append(corners_2)

            cv2.drawChessboardCorners(image, (num_interior_corners_y, num_interior_corners_x), corners_2, success)
            cv2_imshow(image)  # Colab-friendly display
            cv2.waitKey(500)
        else:
            print("Chessboard not detected in:", image_file)

    if len(object_points) == 0:
        print("No chessboard corners detected in any image!")
        return

    # Camera calibration
    ret, mtx, dist, rvecs, tvecs = cv2.calibrateCamera(object_points,
                                                       image_points,
                                                       gray.shape[::-1],
                                                       None,
                                                       None)

    # Save calibration parameters
    cv_file = cv2.FileStorage('calibration_chessboard_colab.yaml', cv2.FILE_STORAGE_WRITE)
    cv_file.write('K', mtx)
    cv_file.write('D', dist)
    cv_file.release()

    # Load parameters (to verify)
    cv_file = cv2.FileStorage('calibration_chessboard_colab.yaml', cv2.FILE_STORAGE_READ)
    mtx_loaded = cv_file.getNode('K').mat()
    dist_loaded = cv_file.getNode('D').mat()
    cv_file.release()

    print("\nCamera matrix (K):\n", mtx_loaded)
    print("\nDistortion coefficients (D):\n", dist_loaded)

    cv2.destroyAllWindows()

def main():
    calibrate()

if __name__ == '__main__':
    main()

Found images: []
No chessboard corners detected in any image!
